# Week 62, Day 2: Reading independence from a graph: d-separation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w062/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Partial correlations in three structures

Write partial_corr(a, c, b), the correlation of the residuals of a and c after regressing each on b. Then simulate 5,000 samples from each structure and record the correlation of A and C, and their partial correlation given B.

In [ ]:
import numpy as np

def partial_corr(a, c, b):
    B = np.column_stack([np.ones_like(b), b])
    ra = a - B @ np.linalg.lstsq(B, a, rcond=None)[0]
    rc = ...
    return np.corrcoef(ra, rc)[0, 1]

gt = np.random.default_rng(622)
n = 5000
results = {}
a = gt.normal(size=n); b = a + gt.normal(size=n); c = b + gt.normal(size=n)
results["chain"] = (np.corrcoef(a, c)[0, 1], partial_corr(a, c, b))
b = gt.normal(size=n); a = b + gt.normal(size=n); c = b + gt.normal(size=n)
results["fork"] = (np.corrcoef(a, c)[0, 1], partial_corr(a, c, b))
a = gt.normal(size=n); c = gt.normal(size=n); b = a + c + 0.5 * gt.normal(size=n)
results["collider"] = (np.corrcoef(a, c)[0, 1], partial_corr(a, c, b))
for k, (r0, r1) in results.items():
    print(f"{k:9s} corr(A, C) = {r0:+.3f}   corr(A, C | B) = {r1:+.3f}")

In [ ]:
import numpy as np
def test_partial():
    "partial_corr removes the linear effect of the conditioning variable"
    g = np.random.default_rng(0); z = g.normal(size=5000); a = z + g.normal(size=5000); c = z + g.normal(size=5000)
    assert abs(partial_corr(a, c, z)) < 0.05 and abs(np.corrcoef(a, c)[0, 1] - 0.5) < 0.05
def test_patterns():
    "chain and fork: dependent until B is given; collider: independent until B is given"
    for k in ("chain", "fork"):
        assert abs(results[k][0]) > 0.2 and abs(results[k][1]) < 0.05
    assert abs(results["collider"][0]) < 0.05 and abs(results["collider"][1]) > 0.2

test_partial()
test_patterns()
print("All checks passed")

### Exercise: A d-separation test

Complete dsep(X, Y, Z) for the Day 1 network: find the ancestors of X, Y and Z, moralise, remove Z, and search for a path from X to Y.

In [ ]:
parents = {"S": [], "A": [], "U": [], "R": ["S", "A", "U"], "P": ["U"], "B": ["R"]}

def ancestors(nodes_):
    out, stack = set(nodes_), list(nodes_)
    while stack:
        for p in parents[stack.pop()]:
            if p not in out:
                out.add(p); stack.append(p)
    return out

def dsep(X, Y, Z):
    keep = ancestors(X | Y | Z)
    edges = set()
    for v in keep:
        ps = [p for p in parents[v] if p in keep]
        for p in ps:
            edges.add(frozenset((p, v)))
        for i in range(len(ps)):                       # marry parents of a common child
            for j in range(i + 1, len(ps)):
                edges.add(...)
    reach, stack = set(X), list(X)
    while stack:
        u = stack.pop()
        for e in edges:
            if u in e:
                (w,) = e - {u}
                if ... :
                    reach.add(w); stack.append(w)
    return not (reach & Y)

for q in [({"S"}, {"U"}, set()), ({"S"}, {"U"}, {"R"}), ({"S"}, {"U"}, {"B"}), ({"P"}, {"B"}, {"R"})]:
    print(f"{sorted(q[0])} independent of {sorted(q[1])} given {sorted(q[2]) or 'nothing'}: {dsep(*q)}")

In [ ]:
import numpy as np
def test_cases():
    "dsep gives the right answers on the experiment network of Day 1"
    assert dsep({"S"}, {"U"}, set()) and not dsep({"S"}, {"U"}, {"R"})
    assert not dsep({"S"}, {"U"}, {"B"}) and dsep({"S"}, {"P"}, set())
    assert dsep({"P"}, {"B"}, {"R"}) and not dsep({"P"}, {"B"}, set())

test_cases()
print("All checks passed")

### Exercise: Common input and selection

Neurons X and Y receive a common input I but are not connected. Compute their correlation and their partial correlation given I. Then make X and Y independent, drive a target neuron T by both, and compute the correlation of X and Y on all trials and on the trials where T is in its top 10%.

In [ ]:
gn = np.random.default_rng(6221)
n = 20000
I = gn.normal(size=n)
X = 0.6 * I + gn.normal(size=n)
Y = 0.6 * I + gn.normal(size=n)
r_fork, r_fork_given = np.corrcoef(X, Y)[0, 1], partial_corr(X, Y, I)

X2, Y2 = gn.normal(size=n), gn.normal(size=n)
T = X2 + Y2 + 0.5 * gn.normal(size=n)
burst = T > np.percentile(T, 90)
r_all = np.corrcoef(X2, Y2)[0, 1]
r_selected = ...
print(f"common input: corr(X, Y) = {r_fork:+.3f}, given the input {r_fork_given:+.3f}")
print(f"common target: corr(X, Y) on all trials {r_all:+.3f}, on burst trials {r_selected:+.3f}")

In [ ]:
import numpy as np
def test_fork():
    "a shared input correlates two unconnected neurons, and conditioning on it removes the correlation"
    assert r_fork > 0.2 and abs(r_fork_given) < 0.05
def test_selection():
    "two independent neurons become negatively correlated on trials selected for a burst in their common target"
    assert abs(r_all) < 0.05 and r_selected < -0.15

test_fork()
test_selection()
print("All checks passed")